# Introduction to LangGraph

## Tutorials Covered:
1. What is LangGraph?
2. LangGraph vs Traditional Approaches
3. Installation and Setup
4. Basic Concepts

## 1. What is LangGraph?

Learning objectives:
- Understand what LangGraph is
- Learn why stateful applications are important
- Explore use cases for LangGraph

In [ ]:
# Tutorial 1: What is LangGraph?

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# Define a simple state structure
class State(TypedDict):
    value: int
    message: str

# Define node functions
def start_node(state):
    print("Starting the graph")
    return {"value": 1, "message": "Initialized"}

def processing_node(state):
    print(f"Processing: current value is {state['value']}")
    new_value = state['value'] * 2
    return {"value": new_value, "message": f"Doubled to {new_value}"}

def end_node(state):
    print(f"Ending: final value is {state['value']}, message: {state['message']}")
    return state

print("LangGraph allows us to build stateful, multi-step applications.")
print("It provides a way to define graphs with conditional logic and state management.")

# Create a simple example
builder = StateGraph(State)
builder.add_node("start", start_node)
builder.add_node("process", processing_node)
builder.add_node("end", end_node)

# Add edges
builder.add_edge("start", "process")
builder.add_edge("process", "end")
builder.set_entry_point("start")
builder.set_finish_point("end")

# Compile the graph
graph = builder.compile()

# Run the graph
result = graph.invoke({})
print(f"Final result: {result}")

## 2. LangGraph vs Traditional Approaches

Learning objectives:
- Compare LangGraph with simple LLM chains
- Understand advantages of graph-based approaches
- Identify when to use LangGraph vs other tools

In [ ]:
# Tutorial 2: LangGraph vs Traditional Approaches

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from typing import TypedDict

# First, let's look at a traditional approach using LangChain sequence
print("=== Traditional Approach with LangChain ===")

# Define a simple prompt template
traditional_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "What is {input}? Provide a brief explanation.")
])

# Create a simple chain
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
traditional_chain = traditional_prompt | llm

# Run the traditional approach
try:
    # We'll simulate a simple call without API key
    print("Traditional approach: Simple sequence of operations")
    print("1. Format prompt")
    print("2. Call LLM")
    print("3. Return response")
    print("Limitation: No conditional logic, no complex state management")
except Exception as e:
    print(f"API Error (expected without key): {e}")

print("\n=== LangGraph Approach ===")

# Define state for our graph
class ComparisonState(TypedDict):
    input: str
    response: str
    processed: bool
    retries: int

# Define nodes for our graph
def format_input_node(state):
    print(f"Formatting input: {state['input']}")
    return state

def call_llm_node(state):
    print(f"Calling LLM with input: {state['input']}")
    # Simulating a response instead of making an API call
    response = f"Response to '{state['input']}' would come from LLM here"
    return {"response": response, "processed": True}

def validation_node(state):
    print(f"Validating response: {state['response']}")
    # Simple validation - in real scenarios, this could check quality, safety, etc.
    if len(state['response']) > 0:
        return {"retries": 0}
    else:
        retries = state.get('retries', 0) + 1
        return {"retries": retries}

# Build the graph
builder = StateGraph(ComparisonState)
builder.add_node("format", format_input_node)
builder.add_node("call_llm", call_llm_node)
builder.add_node("validate", validation_node)

# Add edges
builder.add_edge("format", "call_llm")
builder.add_edge("call_llm", "validate")

# Set entry and finish points
builder.set_entry_point("format")
builder.set_finish_point("validate")

graph = builder.compile()

# Run the graph
try:
    result = graph.invoke({"input": "langgraph", "response": "", "processed": False, "retries": 0})
    print(f"\nGraph result: {result}")
    print("\nAdvantages of LangGraph:")
    print("- State management")
    print("- Conditional logic")
    print("- Retry mechanisms")
    print("- Complex workflows")
    print("- Ability to handle failures gracefully")
except Exception as e:
    print(f"Error running graph: {e}")

## 3. Installation and Setup

Learning objectives:
- Install LangGraph and dependencies
- Set up development environment
- Verify installation

In [ ]:
# Tutorial 3: Installation and Setup

import sys
import subprocess
import importlib

print("=== Installing LangGraph ===\n")

# Check if we're in a virtual environment
in_venv = sys.prefix != sys.base_prefix
print(f"Running in virtual environment: {in_venv}")

# List of packages to install
packages = [
    "langgraph",
    "langchain-openai",  # for OpenAI integration examples
    "langchain-community"  # for general langchain utilities
]

# Install packages
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"{package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"Successfully installed {package}")

print("\n=== Verifying Installation ===\n")

# Test imports
try:
    from langgraph.graph import StateGraph, END
    print("✓ Successfully imported StateGraph and END from langgraph.graph")
    
    from typing import TypedDict, Annotated
    print("✓ Successfully imported TypedDict and Annotated from typing")
    
    import operator
    print("✓ Successfully imported operator module")
    
    print("\n=== Creating a Simple Test Graph ===\n")
    
    # Define a simple state
    class SimpleState(TypedDict):
        count: int
        message: str
    
    # Define a simple node function
    def increment_node(state):
        new_count = state['count'] + 1
        return {'count': new_count, 'message': f'Count is now {new_count}'}
    
    # Create and compile a simple graph
    builder = StateGraph(SimpleState)
    builder.add_node("increment", increment_node)
    builder.add_edge("__start__", "increment")
    builder.add_edge("increment", "__end__")
    
    graph = builder.compile()
    
    # Test the graph
    result = graph.invoke({'count': 0, 'message': 'Initial state'})
    print(f"Test graph result: {result}")
    
    print("\n✓ Installation and basic functionality test passed!")
    
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("Installation may have failed or environment issues exist")
except Exception as e:
    print(f"✗ Error during test: {e}")

print("\n=== Setup Complete ===")
print("You're now ready to work with LangGraph!")

## 4. Basic Concepts

Learning objectives:
- Understand nodes, edges, state, and graph construction
- Learn fundamental building blocks of LangGraph
- Create your first simple graph

In [ ]:
# Tutorial 4: Basic Concepts

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

print("=== Understanding Basic LangGraph Concepts ===\n")

# 1. STATE: Defines the data structure that flows through the graph
class BasicConceptsState(TypedDict):
    counter: int
    message: str
    history: list[str]

print("1. STATE - The data structure that maintains information across nodes")
print("   Our state includes:")
print("   - counter: integer that tracks how many times we've processed")
print("   - message: string that holds a message")
print("   - history: list that keeps track of previous messages\n")

# 2. NODES: Functions that perform operations and update the state
def increment_node(state):
    print(f"   Inside increment_node: current counter = {state['counter']}")
    new_counter = state['counter'] + 1
    new_history = state['history'] + [f"Incremented to {new_counter}"]
    return {'counter': new_counter, 'history': new_history}

def message_node(state):
    print(f"   Inside message_node: current message = '{state['message']}'")
    new_message = f"Processed item #{state['counter']}: {state['message']}"
    new_history = state['history'] + [f"Added message for item #{state['counter']}"]
    return {'message': new_message, 'history': new_history}

def finalize_node(state):
    print(f"   Inside finalize_node: finalizing with counter={state['counter']}")
    final_message = f"Final result after {state['counter']} increments: {state['message']}"
    new_history = state['history'] + ["Graph completed"]
    return {'message': final_message, 'history': new_history}

print("2. NODES - Functions that transform the state")
print("   - increment_node: increases the counter")
print("   - message_node: creates a message based on the counter")
print("   - finalize_node: creates a final message\n")

# 3. EDGES: Connections between nodes
def create_basic_graph():
    print("3. EDGES - Connections that define the flow of execution")
    print("   Creating a simple linear graph: start -> increment -> message -> finalize -> end\n")
    
    builder = StateGraph(BasicConceptsState)
    
    # Add nodes
    builder.add_node("increment", increment_node)
    builder.add_node("message", message_node)
    builder.add_node("finalize", finalize_node)
    
    # Add edges (defines the flow)
    builder.add_edge("__start__", "increment")  # Start with increment node
    builder.add_edge("increment", "message")    # Then go to message node
    builder.add_edge("message", "finalize")     # Then go to finalize node
    builder.add_edge("finalize", "__end__")     # End after finalize
    
    # Compile the graph
    graph = builder.compile()
    return graph

# 4. GRAPH CONSTRUCTION: Bringing everything together
print("4. GRAPH CONSTRUCTION - Combining state, nodes, and edges\n")

# Create and run the graph
basic_graph = create_basic_graph()

initial_state = {
    "counter": 0,
    "message": "Hello, LangGraph!",
    "history": ["Graph started"]
}

print("Running the graph with initial state:", initial_state)
print("\nExecution trace:")
result = basic_graph.invoke(initial_state)

print(f"\nFinal result: {result}")

print("\n=== Summary of Basic Concepts ===")
print("- State: Maintains data across node executions")
print("- Nodes: Functions that transform state")
print("- Edges: Define execution flow between nodes")
print("- Graph: Combines all elements into a runnable workflow")